# Notebook Colab: “Tres tipos de aprendizaje a nivel de sistema”


# 1. Meta-learning simplificado: “aprender a combinar modelos”

Ejemplo con stacking, donde un meta-modelo aprende a combinar predicciones de varios modelos base.

Idea clave: el modelo no elige un único algoritmo; aprende a combinar varios.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Dataset
X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Modelos base
base_models = [
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("gb", GradientBoostingClassifier(random_state=42)),
    ("svc", SVC(probability=True, random_state=42)),
]

# Meta-modelo
meta_model = LogisticRegression(max_iter=1000)

# Stacking
stacking_model = StackingClassifier(
    estimators=base_models, final_estimator=meta_model, cv=5
)

stacking_model.fit(X_train, y_train)
y_pred = stacking_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.958041958041958
              precision    recall  f1-score   support

           0       0.96      0.92      0.94        53
           1       0.96      0.98      0.97        90

    accuracy                           0.96       143
   macro avg       0.96      0.95      0.95       143
weighted avg       0.96      0.96      0.96       143



# 2. Transfer learning: usar un modelo preentrenado

Ejemplo con MobileNetV2 para clasificar imágenes de gatos y perros usando conocimiento aprendido previamente en ImageNet.

Idea clave: el modelo reutiliza conocimiento aprendido en un dominio grande y lo adapta a una nueva tarea.


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models

# Cargar dataset pequeño
(ds_train, ds_test), ds_info = tfds.load(
    "cats_vs_dogs",
    split=["train[:10%]", "train[10%:12%]"],
    as_supervised=True,
    with_info=True,
)

IMG_SIZE = 160
BATCH_SIZE = 32


def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label


ds_train = ds_train.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_test.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Modelo base preentrenado
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet"
)

base_model.trainable = False

# Nuevo clasificador
model = models.Sequential(
    [
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ]
)

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history = model.fit(ds_train, validation_data=ds_test, epochs=3)

Epoch 1/3
73/73 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.9579 - loss: 0.1109 - val_accuracy: 0.9785 - val_loss: 0.0583
Epoch 2/3
73/73 ━━━━━━━━━━━━━━━━━━━━ 66s 898ms/step - accuracy: 0.9862 - loss: 0.0426 - val_accuracy: 0.9828 - val_loss: 0.0475
Epoch 3/3
73/73 ━━━━━━━━━━━━━━━━━━━━ 67s 913ms/step - accuracy: 0.9940 - loss: 0.0217 - val_accuracy: 0.9871 - val_loss: 0.0415


#3. Federated learning: aprender sin centralizar datos

Ejemplo con Flower, simulando varios clientes con datos locales.

Idea clave: cada cliente entrena localmente y solo comparte actualizaciones del modelo, no sus datos crudos.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import copy

# Define Federated Learning parameters
NUM_CLIENTS = 5
LOCAL_EPOCHS = 2  # Number of local epochs each client performs
BATCH_SIZE = 32  # Batch size for local client training
NUM_ROUNDS = 5  # Number of communication rounds
GLOBAL_LR = 0.01  # Learning rate for clients

# Device configuration
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


# 1. Model Definition (Simple CNN for MNIST)
class SimpleCNN(nn.Module):
    def __init__(self) -> None:
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)
        self.fc = nn.Linear(320, 10)  # 20 channels * 4x4 image size after pooling

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(-1, 320)  # Flatten the tensor
        x = self.fc(x)
        return x


# 2. Dataset Preparation (MNIST, distributed among clients)
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
)

print("Downloading MNIST dataset...")
train_dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST("./data", train=False, download=True, transform=transform)

# Partitioning data among clients (non-IID for demonstration purposes, or IID if preferred)
# Here, we'll do IID partitioning for simplicity.
partition_size = len(train_dataset) // NUM_CLIENTS
lengths = [partition_size] * NUM_CLIENTS
# Adjust the last partition to absorb any remaining data if len(train_dataset) is not perfectly divisible
lengths[-1] += len(train_dataset) - sum(lengths)

# Using torch.Generator for reproducible splits
client_datasets = torch.utils.data.random_split(
    train_dataset, lengths, generator=torch.Generator().manual_seed(42)
)

# Create DataLoaders for each client
client_trainloaders = []
for i, ds in enumerate(client_datasets):
    if len(ds) == 0:
        print(
            f"Warning: Client {i} has an empty dataset partition. Consider reducing NUM_CLIENTS or using a larger dataset."
        )
        continue
    client_trainloaders.append(DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True))

# Global test set for server evaluation
global_testloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


# 3. Client Training Function
def client_update(model, trainloader, epochs, lr, device):
    model.train()
    model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        for batch_idx, (data, target) in enumerate(trainloader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
    return model.state_dict()


# 4. Server Aggregation Function (Federated Averaging)
def federated_averaging(global_model, client_models_states):
    # Initialize an empty state dict for aggregated model
    aggregated_state_dict = copy.deepcopy(global_model.state_dict())
    for key in aggregated_state_dict.keys():
        aggregated_state_dict[key] = torch.zeros_like(aggregated_state_dict[key])

    # Sum up all client model parameters
    for client_state in client_models_states:
        for key in aggregated_state_dict.keys():
            aggregated_state_dict[key] += client_state[key]

    # Average them
    for key in aggregated_state_dict.keys():
        aggregated_state_dict[key] = aggregated_state_dict[key] / len(
            client_models_states
        )

    global_model.load_state_dict(aggregated_state_dict)
    return global_model


# 5. Global Model Evaluation Function
def evaluate_global_model(model, testloader, device):
    model.eval()
    model.to(device)
    loss = 0.0
    correct = 0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for data, target in testloader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    loss /= len(testloader.dataset)
    accuracy = correct / len(testloader.dataset)
    return loss, accuracy


# 6. Federated Learning Simulation Loop
print(
    f"Starting Federated Learning Simulation with {NUM_CLIENTS} clients and {NUM_ROUNDS} rounds..."
)

# Initialize global model
global_model = SimpleCNN().to(DEVICE)
initial_global_state = copy.deepcopy(global_model.state_dict())

for round_num in range(NUM_ROUNDS):
    print(f"\n--- Communication Round {round_num + 1}/{NUM_ROUNDS} ---")

    # Simulate client participation
    client_model_states = []
    for i in range(NUM_CLIENTS):
        print(f"  Client {i+1} training...")
        # Each client starts with the current global model
        client_model = SimpleCNN()
        client_model.load_state_dict(initial_global_state)  # Load global model state

        # Train the client's model locally
        updated_state = client_update(
            client_model, client_trainloaders[i], LOCAL_EPOCHS, GLOBAL_LR, DEVICE
        )
        client_model_states.append(updated_state)

    # Server aggregates models
    print("  Server aggregating models...")
    global_model = federated_averaging(global_model, client_model_states)
    initial_global_state = copy.deepcopy(
        global_model.state_dict()
    )  # Update for next round

    # Evaluate global model
    global_loss, global_accuracy = evaluate_global_model(
        global_model, global_testloader, DEVICE
    )
    print(
        f"  Global Model Test Loss: {global_loss:.4f}, Global Model Test Accuracy: {global_accuracy:.4f}"
    )

print("\nFederated Learning Simulation Completed.")

Using device: cuda:0
Starting Federated Learning Simulation with 5 clients and 5 rounds...

--- Communication Round 1/5 ---
  Client 1 training...
  Client 2 training...
  Client 3 training...
  Client 4 training...
  Client 5 training...
  Server aggregating models...
  Global Model Test Loss: 0.0060, Global Model Test Accuracy: 0.9452

--- Communication Round 2/5 ---
  Client 1 training...
  Client 2 training...
  Client 3 training...
  Client 4 training...
  Client 5 training...
  Server aggregating models...
  Global Model Test Loss: 0.0036, Global Model Test Accuracy: 0.9655

--- Communication Round 3/5 ---
  Client 1 training...
  Client 2 training...
  Client 3 training...
  Client 4 training...
  Client 5 training...
  Server aggregating models...
  Global Model Test Loss: 0.0027, Global Model Test Accuracy: 0.9749

--- Communication Round 4/5 ---
  Client 1 training...
  Client 2 training...
  Client 3 training...
  Client 4 training...
  Client 5 training...
  Server aggregat

#Comparación

| Tipo               | Pregunta central                              | Ejemplo                     |
| ------------------ | --------------------------------------------- | --------------------------- |
| Meta-learning      | ¿Cómo aprender a aprender o combinar modelos? | Stacking                    |
| Transfer learning  | ¿Cómo reutilizar conocimiento previo?         | MobileNetV2 preentrenado    |
| Federated learning | ¿Cómo aprender sin compartir datos?           | FedAvg con clientes locales |
